In [ ]:
import numpy as np 
import pandas as pd 
from pathlib import Path

In [ ]:
ssd_chemicals = pd.read_csv('./external_data/ssd_chemicals.tsv', sep='\t', dtype=str)
print(ssd_chemicals.shape)
ssd_chemicals.head()

# Acute SSD results

In [ ]:
ssd_acute_chemicals = ssd_chemicals[ssd_chemicals['Type']=='Acute']
print(ssd_acute_chemicals.shape)
ssd_acute_chemicals.head()

In [ ]:
ssd_acute_chemicals_list = set(ssd_acute_chemicals['Chemical']) - {''}
print(len(ssd_acute_chemicals_list))

Output folder for acute SSD computation

In [ ]:
home = Path.cwd()
acute_ssd_data_path = home.joinpath("ssd_output", "acute")
print(acute_ssd_data_path)

In [ ]:
# Here we will create dictionaries of dataframes which will be cleaned and concatenated later
acute_gof_files = {}
acute_gof_pval_files = {}
acute_hc5_data_files = {}

for k in sorted(ssd_acute_chemicals_list):
    # read goodness of fit test results
    acute_gof_files[k] = pd.read_csv(acute_ssd_data_path.joinpath(f"{k}", f"{k}_gof.csv"), dtype=str)
    acute_gof_files[k] = acute_gof_files[k].replace(np.nan, '', regex=True)
    acute_gof_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    acute_gof_files[k].insert(0, 'cas', k)

    # read goodness of fit test results with pvalues
    acute_gof_pval_files[k] = pd.read_csv(acute_ssd_data_path.joinpath(f"{k}", f"{k}_gof_pvalues.csv"), dtype=str)
    acute_gof_pval_files[k] = acute_gof_pval_files[k].replace(np.nan, '', regex=True)
    acute_gof_pval_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    acute_gof_pval_files[k].insert(0, 'cas', k)

    # read HC05 data
    acute_hc5_data_files[k] = pd.read_csv(acute_ssd_data_path.joinpath(f"{k}", f"{k}_hc5_data.csv"), dtype=str)
    acute_hc5_data_files[k] = acute_hc5_data_files[k].replace(np.nan, '', regex=True)
    acute_hc5_data_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    acute_hc5_data_files[k].insert(0, 'cas', k)

In [ ]:
len(acute_gof_files.keys())

In [ ]:
# example
acute_gof_files['50-00-0']

### Goodness-of-fit table

In [ ]:
acute_gof_files_df = pd.concat(list(acute_gof_files.values()))
print(acute_gof_files_df.shape)
acute_gof_files_df.head()

In [ ]:
print(len(set(acute_gof_files_df['cas'])))

In [ ]:
acute_gof_files_df.to_csv("./acute_gof_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
bestfit_acute_gof_files_df = acute_gof_files_df[acute_gof_files_df['delta'].apply(float)<=0]
print(bestfit_acute_gof_files_df.shape)
bestfit_acute_gof_files_df.head()

In [ ]:
# check: unique delta should be 0 and number of rows should match the number of unique chemicals
print(set(bestfit_acute_gof_files_df['delta']))
print(len(bestfit_acute_gof_files_df['cas'])==len(ssd_acute_chemicals_list))

In [ ]:
bestfit_acute_gof_files_df.to_csv("./acute_bestfit_gof_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
cas_bestfit_pairs = dict(zip(bestfit_acute_gof_files_df['cas'], bestfit_acute_gof_files_df['dist']))
print(len(cas_bestfit_pairs))
print("Best fit for '50-00-0' is ", cas_bestfit_pairs['50-00-0'])

### Goodness-of-fit table with pvalues

In [ ]:
acute_gof_pval_files_df = pd.concat(list(acute_gof_pval_files.values()))
print(acute_gof_pval_files_df.shape)
acute_gof_pval_files_df.head()

In [ ]:
print(len(set(acute_gof_pval_files_df['cas'])))

In [ ]:
acute_gof_pval_files_df.to_csv("./acute_gof_pvalues_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
bestfit_acute_gof_pval_files_df = acute_gof_pval_files_df[acute_gof_pval_files_df['delta'].apply(float)<=0]
print(bestfit_acute_gof_pval_files_df.shape)
bestfit_acute_gof_pval_files_df.head()

In [ ]:
# check: unique delta should be 0 and number of rows should match the number of unique chemicals
print(set(bestfit_acute_gof_pval_files_df['delta']))
print(len(bestfit_acute_gof_pval_files_df['cas'])==len(ssd_acute_chemicals_list))

In [ ]:
bestfit_acute_gof_pval_files_df.to_csv("./acute_bestfit_gof_pvalues_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

### HC05 table

In [ ]:
acute_hc5_data_files_df = pd.concat(list(acute_hc5_data_files.values()))
print(acute_hc5_data_files_df.shape)
acute_hc5_data_files_df.head()

In [ ]:
print(len(set(acute_hc5_data_files_df['cas'])))

In [ ]:
acute_hc5_data_files_df['is_bestfit'] = acute_hc5_data_files_df.apply(lambda row: 'Yes' if row['dist']==cas_bestfit_pairs[row['cas']] else 'No', axis=1)
print(acute_hc5_data_files_df.shape)
acute_hc5_data_files_df.head()

In [ ]:
# check
acute_hc5_data_files_df[acute_hc5_data_files_df['is_bestfit']=='Yes'].shape[0] == len(ssd_acute_chemicals_list)

In [ ]:
acute_hc5_data_files_df.to_csv("./acute_hc05_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
acute_hc5_data_files_df = acute_hc5_data_files_df[acute_hc5_data_files_df['dist']=='average']
print(acute_hc5_data_files_df.shape)
acute_hc5_data_files_df.sort_values(by=['est'], inplace=True)
acute_hc5_data_files_df.head()

In [ ]:
acute_hc5_data_files_df = acute_hc5_data_files_df.merge(ssd_acute_chemicals, left_on='cas', right_on='Chemical', how='inner')
print(acute_hc5_data_files_df.shape)
acute_hc5_data_files_df.sort_values(by=['est'], inplace=True)
acute_hc5_data_files_df.head()

In [ ]:
acute_hc5_data_files_df.to_csv("./acute_averaged_hc05_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

# Chronic SSD results

In [ ]:
ssd_chronic_chemicals = ssd_chemicals[ssd_chemicals['Type']=='chronic']
print(ssd_chronic_chemicals.shape)
ssd_chronic_chemicals.head()

In [ ]:
ssd_chronic_chemicals_list = set(ssd_chronic_chemicals['Chemical']) - {''}
print(len(ssd_chronic_chemicals_list))

Output folder for chronic SSD computation

In [ ]:
home = Path.cwd()
chronic_ssd_data_path = home.joinpath("ssd_output", "chronic")
print(chronic_ssd_data_path)

In [ ]:
# Here we will create dictionaries of dataframes which will be cleaned and concatenated later
chronic_gof_files = {}
chronic_gof_pval_files = {}
chronic_hc5_data_files = {}

for k in sorted(ssd_chronic_chemicals_list):
    # read goodness of fit test results
    chronic_gof_files[k] = pd.read_csv(chronic_ssd_data_path.joinpath(f"{k}", f"{k}_gof.csv"), dtype=str)
    chronic_gof_files[k] = chronic_gof_files[k].replace(np.nan, '', regex=True)
    chronic_gof_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    chronic_gof_files[k].insert(0, 'cas', k)

    # read goodness of fit test results with pvalues
    chronic_gof_pval_files[k] = pd.read_csv(chronic_ssd_data_path.joinpath(f"{k}", f"{k}_gof_pvalues.csv"), dtype=str)
    chronic_gof_pval_files[k] = chronic_gof_pval_files[k].replace(np.nan, '', regex=True)
    chronic_gof_pval_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    chronic_gof_pval_files[k].insert(0, 'cas', k)

    # read HC05 data
    chronic_hc5_data_files[k] = pd.read_csv(chronic_ssd_data_path.joinpath(f"{k}", f"{k}_hc5_data.csv"), dtype=str)
    chronic_hc5_data_files[k] = chronic_hc5_data_files[k].replace(np.nan, '', regex=True)
    chronic_hc5_data_files[k].drop(columns=['Unnamed: 0'], inplace=True)
    chronic_hc5_data_files[k].insert(0, 'cas', k)

In [ ]:
len(chronic_gof_files.keys())

In [ ]:
# example
chronic_gof_files['50-00-0']

### Goodness-of-fit table

In [ ]:
chronic_gof_files_df = pd.concat(list(chronic_gof_files.values()))
print(chronic_gof_files_df.shape)
chronic_gof_files_df.head()

In [ ]:
print(len(set(chronic_gof_files_df['cas'])))

In [ ]:
chronic_gof_files_df.to_csv("./chronic_gof_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
bestfit_chronic_gof_files_df = chronic_gof_files_df[chronic_gof_files_df['delta'].apply(float)<=0]
print(bestfit_chronic_gof_files_df.shape)
bestfit_chronic_gof_files_df.head()

In [ ]:
# check: unique delta should be 0 and number of rows should match the number of unique chemicals
print(set(bestfit_chronic_gof_files_df['delta']))
print(len(bestfit_chronic_gof_files_df['cas'])==len(ssd_chronic_chemicals_list))

In [ ]:
bestfit_chronic_gof_files_df.to_csv("./chronic_bestfit_gof_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
cas_bestfit_pairs_chronic = dict(zip(bestfit_chronic_gof_files_df['cas'], bestfit_chronic_gof_files_df['dist']))
print(len(cas_bestfit_pairs_chronic))
print("Best fit for '50-00-0' is ", cas_bestfit_pairs_chronic['50-00-0'])

### Goodness-of-fit table with pvalues

In [ ]:
chronic_gof_pval_files_df = pd.concat(list(chronic_gof_pval_files.values()))
print(chronic_gof_pval_files_df.shape)
chronic_gof_pval_files_df.head()

In [ ]:
print(len(set(chronic_gof_pval_files_df['cas'])))

In [ ]:
chronic_gof_pval_files_df.to_csv("./chronic_gof_pvalues_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
bestfit_chronic_gof_pval_files_df = chronic_gof_pval_files_df[chronic_gof_pval_files_df['delta'].apply(float)<=0]
print(bestfit_chronic_gof_pval_files_df.shape)
bestfit_chronic_gof_pval_files_df.head()

In [ ]:
# check: unique delta should be 0 and number of rows should match the number of unique chemicals
print(set(bestfit_chronic_gof_pval_files_df['delta']))
print(len(bestfit_chronic_gof_pval_files_df['cas'])==len(ssd_chronic_chemicals_list))

In [ ]:
bestfit_chronic_gof_pval_files_df.to_csv("./chronic_bestfit_gof_pvalues_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

### HC05 table

In [ ]:
chronic_hc5_data_files_df = pd.concat(list(chronic_hc5_data_files.values()))
print(chronic_hc5_data_files_df.shape)
chronic_hc5_data_files_df.head()

In [ ]:
print(len(set(chronic_hc5_data_files_df['cas'])))

In [ ]:
chronic_hc5_data_files_df['is_bestfit'] = chronic_hc5_data_files_df.apply(lambda row: 'Yes' if row['dist']==cas_bestfit_pairs_chronic[row['cas']] else 'No', axis=1)
print(chronic_hc5_data_files_df.shape)
chronic_hc5_data_files_df.head()

In [ ]:
# check
chronic_hc5_data_files_df[chronic_hc5_data_files_df['is_bestfit']=='Yes'].shape[0] == len(ssd_chronic_chemicals_list)

In [ ]:
chronic_hc5_data_files_df.to_csv("./chronic_hc05_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')

In [ ]:
chronic_hc5_data_files_df = chronic_hc5_data_files_df[chronic_hc5_data_files_df['dist']=='average']
print(chronic_hc5_data_files_df.shape)
chronic_hc5_data_files_df.sort_values(by=['est'], inplace=True)
chronic_hc5_data_files_df.head()

In [ ]:
chronic_hc5_data_files_df = chronic_hc5_data_files_df.merge(ssd_chronic_chemicals, left_on='cas', right_on='Chemical', how='inner')
print(chronic_hc5_data_files_df.shape)
chronic_hc5_data_files_df.sort_values(by=['est'], inplace=True)
chronic_hc5_data_files_df.head()

In [ ]:
chronic_hc5_data_files_df.to_csv("./chronic_averaged_hc05_all_chemicals.tsv", sep='\t', index=None, encoding='utf-8')